# 🏆 Notebook 14: Capstone — Fine-tune & Serve Gemma 4

Load a quantized Gemma 4 model, fine-tune with LoRA, evaluate, and serve locally. All within the 48GB memory budget.

**Prerequisites:** All previous notebooks

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Gemma 4 Architecture Overview

Gemma 4 is Google's latest open model family. Key features:
- RoPE positional encoding
- GQA (Grouped Query Attention)
- GeGLU FFN variant
- Logit soft-capping

## Memory Budget for 12B 4-bit

| Component | Memory |
|-----------|--------|
| Model weights (4-bit) | ~7 GB |
| LoRA adapters | ~0.1 GB |
| Optimizer state | ~0.4 GB |
| KV-cache (8K context) | ~1 GB |
| Activations | ~2 GB |
| **Total** | **~10.5 GB** |
| Available (48GB - 4GB OS) | 44 GB |
| **Fits?** | ✅ Yes |

## Loading Gemma 4

⚠️ This cell requires downloading a model from Hugging Face (~7GB). Uncomment to run.

In [ ]:
# Uncomment to load Gemma 4 (requires HF access token for gated models)
# from mlx_lm import load, generate
# 
# model, tokenizer = load('mlx-community/gemma-2-2b-it-4bit')  # Use smaller variant for demo
# 
# # Verify it fits in memory
# import mlx.core as mx
# mem_gb = mx.metal.get_active_memory() / 1e9
# print(f'Memory after loading: {mem_gb:.1f} GB')
# 
# # Baseline generation
# response = generate(model, tokenizer, prompt='What is a transformer?', max_tokens=100)
# print(response)

print('To load Gemma 4:')
print('  from mlx_lm import load, generate')
print('  model, tokenizer = load("mlx-community/gemma-2-2b-it-4bit")')
print('  response = generate(model, tokenizer, prompt="Hello", max_tokens=100)')

## LoRA Fine-tuning

**LoRA** (Low-Rank Adaptation) adds small trainable matrices to frozen model weights:

$$W' = W + \alpha \cdot B \cdot A$$

Where $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times d}$ with rank $r \ll d$.

💡 For a 12B model with rank=8 targeting attention layers: ~0.1% trainable parameters.

In [ ]:
import mlx.core as mx
import mlx.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation."""
    def __init__(self, in_features, out_features, rank=8, alpha=16.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.lora_A = mx.random.normal((in_features, rank)) * 0.01
        self.lora_B = mx.zeros((rank, out_features))
        self.alpha = alpha
        self.rank = rank
        # Freeze the original weights
        self.linear.freeze()
    
    def __call__(self, x):
        base = self.linear(x)
        lora = (x @ self.lora_A @ self.lora_B) * (self.alpha / self.rank)
        return base + lora

# Demo
lora = LoRALinear(256, 256, rank=8)
x = mx.random.normal((1, 10, 256))
out = lora(x)
mx.eval(out)

base_params = 256 * 256
lora_params = 256 * 8 + 8 * 256
print(f'Base params: {base_params:,}')
print(f'LoRA params: {lora_params:,} ({lora_params/base_params*100:.1f}% of base)')
print(f'Output shape: {out.shape} ✅')

## Fine-tuning Execution

⚠️ Actual fine-tuning requires a loaded model and dataset. This shows the training loop structure.

In [ ]:
# Fine-tuning loop template
print('Fine-tuning loop structure:')
print('  1. Load base model (4-bit quantized)')
print('  2. Apply LoRA adapters to target modules (Q, K, V projections)')
print('  3. Prepare dataset (instruction-response pairs)')
print('  4. Training loop:')
print('     - Forward pass through model')
print('     - Compute cross-entropy loss')
print('     - Backward pass (only LoRA params get gradients)')
print('     - Update LoRA params with AdamW')
print('     - Monitor memory with mx.metal.get_active_memory()')
print('  5. Save LoRA adapter weights')
print('  6. Load base + adapter for inference')

## Evaluation & Serving

Compare base vs fine-tuned model outputs, measure perplexity, and serve interactively.

In [ ]:
print('Evaluation metrics:')
print('  - Perplexity on held-out data')
print('  - Side-by-side comparison of base vs fine-tuned outputs')
print('  - Inference speed (tok/s)')
print('  - Memory usage during generation')
print()
print('Serving:')
print('  - Load base model + LoRA adapter')
print('  - Interactive generation with streaming')
print('  - Final performance benchmarks')

## 🎉 Congratulations!

You've completed the entire LLM from Scratch series on Apple Silicon:

1. ✅ Environment & Apple Silicon fundamentals
2. ✅ MLX framework mastery
3. ✅ Math foundations (dot products, softmax, cross-entropy)
4. ✅ Tokenization (character, BPE, tiktoken, SentencePiece)
5. ✅ Embeddings & positional encoding (sinusoidal, RoPE)
6. ✅ Self-attention (single-head, multi-head, GQA)
7. ✅ Transformer architecture (FFN, norms, residuals)
8. ✅ Building & training a GPT from scratch
9. ✅ Apple Silicon training optimizations
10. ✅ Modern architectures (LLaMA, Mistral, Gemma)
11. ✅ Metal custom kernels
12. ✅ Inference optimization (KV-cache, quantization)
13. ✅ Advanced attention (Flash, Paged, Ring)
14. ✅ Local serving (mlx-lm, llama.cpp)
15. ✅ Capstone: Fine-tuning & serving Gemma 4

🎯 You're now ready for AI lab interviews at Google DeepMind, OpenAI, Anthropic, Apple ML, and Meta FAIR!

## 📜 History & Alternatives

### The Evolution of Google's Open Models: From Gemma to Gemma 4

Google's Gemma family represents a strategic shift toward open-weight models, distilling capabilities from the proprietary Gemini series into smaller, publicly available models.

| Year | Model | Team | Key Contribution |
|------|-------|------|-----------------|
| 2023 (Dec) | **Gemini 1.0** | Google DeepMind | Proprietary multimodal model — Ultra/Pro/Nano variants |
| 2024 (Feb) | **Gemma 1** | Google DeepMind | First open-weight Gemma — 2B/7B, distilled from Gemini, RoPE + Multi-Query Attention |
| 2024 (Jun) | **Gemma 2** | Google DeepMind | 2B/9B/27B, knowledge distillation, sliding window + global attention alternation, GQA |
| 2024 (Aug) | **Gemini 1.5 Pro** | Google DeepMind | 1M token context, MoE architecture (proprietary) |
| 2024 (Oct) | **ShieldGemma** | Google DeepMind | Safety-focused classifier built on Gemma — content filtering |
| 2025 (Mar) | **Gemma 3** | Google DeepMind | 1B/4B/12B/27B, multimodal (vision), 128K context, p-RoPE |
| 2025 (Jun) | **Gemini 2.5 Pro** | Google DeepMind | Thinking model, 1M context, best-in-class reasoning |
| 2026 | **Gemma 4** | Google DeepMind | Diffusion-based architecture, MoE with shared experts, advanced reasoning capabilities |

### 💡 Google's Open Model Strategy

Google's approach differs from Meta's (LLaMA) and Mistral's strategies:

| Aspect | Google (Gemma) | Meta (LLaMA) | Mistral |
|--------|---------------|-------------|---------|
| **Source** | Distilled from Gemini | Trained from scratch | Trained from scratch |
| **Sizes** | 1B-27B | 1B-405B | 7B-8x22B |
| **License** | Gemma license (restricted) | Llama license (permissive) | Apache 2.0 |
| **Multimodal** | Yes (Gemma 3+) | Yes (LLaMA 3.2+) | Limited (Pixtral) |
| **MoE** | Gemma 4 | LLaMA 4 | Mixtral |
| **Key advantage** | Gemini distillation | Scale + open community | Efficiency |

### The Gemma Architecture Evolution

```
Gemma 1 (2024):  Standard transformer, MQA, RoPE
    ↓
Gemma 2 (2024):  + Knowledge distillation, sliding window attention, GQA, logit soft-capping
    ↓
Gemma 3 (2025):  + Multimodal (SigLIP vision encoder), p-RoPE, 128K context
    ↓
Gemma 4 (2026):  + Diffusion-based generation, MoE with shared experts, enhanced reasoning
```

### ⚡ Why Gemma 4 Matters

Gemma 4 represents several architectural firsts for open models:
- **Diffusion-based**: Moves beyond autoregressive token-by-token generation
- **MoE with shared experts**: Every token uses a shared expert (always active) plus routed experts — similar to DeepSeek-V3's architecture
- **Reasoning capabilities**: Built-in chain-of-thought and structured reasoning

### Competitive Landscape (2025-2026)

| Model | Params (Active) | Architecture | Open Weight | Reasoning |
|-------|----------------|-------------|-------------|-----------|
| GPT-4o | Unknown | Dense (?) | No | Yes (o1/o3) |
| Claude 3.5 Sonnet | Unknown | Dense (?) | No | Yes |
| Gemini 2.5 Pro | Unknown | MoE | No | Yes (thinking) |
| LLaMA 4 Maverick | ~17B active / 400B total | MoE | Yes | Limited |
| DeepSeek-R1 | 37B active / 671B total | MoE | Yes | Yes (RL-trained) |
| **Gemma 4** | TBD | MoE + Diffusion | Yes | Yes |
| Qwen 3 | Various | Dense + MoE | Yes | Yes (thinking) |

### 🎯 Interview Tip

> *"Gemma 4's shift to diffusion-based generation is significant because it breaks the autoregressive bottleneck — instead of generating one token at a time (sequential, memory-bandwidth-bound), diffusion models can generate multiple tokens in parallel through iterative denoising. Combined with MoE (where only a fraction of parameters are active per token), this creates a model that's both more capable and more efficient at inference time. The shared expert design (from DeepSeek-V3) ensures a baseline of computation for every token while routed experts provide specialization."*